[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/08_attention_core.ipynb)

# 08. Attention core — MHA, MQA, GQA, and MLA

이전 MLA section은 `x → latent → K/V 복원`만 보여줘 DeepSeek MLA의 중요한 **shared compressed KV latent cache와 position/content 분리된 key path**가 빠져 있었다.

이번 버전은 MHA → MQA → GQA의 KV-cache 차이를 먼저 보고, MLA에서 content latent `c_KV`와 decoupled RoPE key를 실제 attention score에 함께 사용하는 구조까지 연결한다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Scaled dot-product attention


In [ ]:
q = torch.tensor(
    [[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]],
    device=device,
)
k = q.clone()
v = torch.tensor(
    [[[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]]],
    device=device,
)

scores = q @ k.transpose(-2, -1)
scores = scores / math.sqrt(q.size(-1))
weights = scores.softmax(dim=-1)
output = weights @ v

print("scores:\n", scores)
print("weights:\n", weights)
print("output:\n", output)


## 2. MHA, MQA, and GQA differ mainly in stored K/V heads

MHA는 query head마다 K/V head가 있고, MQA는 모든 query head가 K/V 하나를 공유하며, GQA는 여러 query heads가 한 KV group을 공유한다. decoding에서는 이 차이가 KV cache size에 직접 영향을 준다.


In [ ]:
batch_size = 1
sequence_length = 6
head_dim = 8
num_query_heads = 4

q = torch.randn(
    batch_size,
    num_query_heads,
    sequence_length,
    head_dim,
    device=device,
)

# MHA: four stored K/V heads.
k_mha = torch.randn_like(q)
v_mha = torch.randn_like(q)
mha_output = F.scaled_dot_product_attention(q, k_mha, v_mha)

# MQA: one stored K/V head, logically shared across four query heads.
k_mqa_stored = torch.randn(
    batch_size, 1, sequence_length, head_dim,
    device=device,
)
v_mqa_stored = torch.randn_like(k_mqa_stored)
k_mqa = k_mqa_stored.expand(-1, num_query_heads, -1, -1)
v_mqa = v_mqa_stored.expand(-1, num_query_heads, -1, -1)
mqa_output = F.scaled_dot_product_attention(q, k_mqa, v_mqa)

# GQA: two stored K/V heads, each shared by two query heads.
num_kv_heads = 2
k_gqa_stored = torch.randn(
    batch_size, num_kv_heads, sequence_length, head_dim,
    device=device,
)
v_gqa_stored = torch.randn_like(k_gqa_stored)
repeat_factor = num_query_heads // num_kv_heads
k_gqa = k_gqa_stored.repeat_interleave(repeat_factor, dim=1)
v_gqa = v_gqa_stored.repeat_interleave(repeat_factor, dim=1)
gqa_output = F.scaled_dot_product_attention(q, k_gqa, v_gqa)

print("MHA stored K elements:", k_mha.numel())
print("MQA stored K elements:", k_mqa_stored.numel())
print("GQA stored K elements:", k_gqa_stored.numel())
print("outputs:", mha_output.shape, mqa_output.shape, gqa_output.shape)


## 3. RoPE helper for MLA positional key/query channels


In [ ]:
def apply_rope(x, positions):
    dim = x.size(-1)
    assert dim % 2 == 0

    pair_index = torch.arange(
        0, dim, 2,
        device=x.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (10000 ** (pair_index / dim))
    angles = positions.float()[:, None] * inverse_frequency[None]

    cos = angles.cos()[None, None]
    sin = angles.sin()[None, None]

    even = x[..., 0::2]
    odd = x[..., 1::2]

    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos

    return torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)


## 4. MLA: cache a low-rank KV latent instead of full per-head K/V

DeepSeek MLA의 핵심 content path는 hidden state를 작은 shared latent `c_KV`로 down-project하고, attention 계산 시 이 latent에서 head-specific content K/V를 up-project하는 것이다. decoding cache에는 full K/V heads 대신 compressed latent를 저장할 수 있다.


In [ ]:
batch_size = 1
sequence_length = 6
model_dim = 32
num_heads = 4
content_head_dim = 6
rope_head_dim = 2
kv_latent_dim = 8

hidden = torch.randn(
    batch_size, sequence_length, model_dim,
    device=device,
)

kv_down_projection = nn.Linear(
    model_dim,
    kv_latent_dim,
    bias=False,
).to(device)
key_content_up = nn.Linear(
    kv_latent_dim,
    num_heads * content_head_dim,
    bias=False,
).to(device)
value_up = nn.Linear(
    kv_latent_dim,
    num_heads * content_head_dim,
    bias=False,
).to(device)

compressed_kv = kv_down_projection(hidden)

key_content = key_content_up(compressed_kv).view(
    batch_size,
    sequence_length,
    num_heads,
    content_head_dim,
).transpose(1, 2)
value = value_up(compressed_kv).view(
    batch_size,
    sequence_length,
    num_heads,
    content_head_dim,
).transpose(1, 2)

print("compressed KV cache:", compressed_kv.shape)
print("expanded content key:", key_content.shape)
print("expanded value:", value.shape)


## 5. MLA decoupled RoPE: position is a separate key/query subspace

MLA는 low-rank content compression과 RoPE를 그대로 섞으면 cache absorption이 어려워지므로, position-dependent RoPE channels를 content key와 분리한다. query도 content part와 RoPE part를 만든 뒤 concatenated Q/K로 score를 계산한다.


In [ ]:
query_content_projection = nn.Linear(
    model_dim,
    num_heads * content_head_dim,
    bias=False,
).to(device)
query_rope_projection = nn.Linear(
    model_dim,
    num_heads * rope_head_dim,
    bias=False,
).to(device)
key_rope_projection = nn.Linear(
    model_dim,
    rope_head_dim,
    bias=False,
).to(device)

query_content = query_content_projection(hidden).view(
    batch_size, sequence_length, num_heads, content_head_dim
).transpose(1, 2)
query_rope = query_rope_projection(hidden).view(
    batch_size, sequence_length, num_heads, rope_head_dim
).transpose(1, 2)

# One positional key is shared across heads, then expanded logically.
key_rope_shared = key_rope_projection(hidden)
key_rope_shared = key_rope_shared[:, None].expand(
    -1, num_heads, -1, -1
)

positions = torch.arange(sequence_length, device=device)
query_rope = apply_rope(query_rope, positions)
key_rope = apply_rope(key_rope_shared, positions)

query = torch.cat([query_content, query_rope], dim=-1)
key = torch.cat([key_content, key_rope], dim=-1)

scores = qk_scores = query @ key.transpose(-2, -1)
scores = scores / math.sqrt(content_head_dim + rope_head_dim)
weights = scores.softmax(dim=-1)
mla_output = weights @ value

print("MLA query:", query.shape)
print("MLA key:", key.shape)
print("MLA output:", mla_output.shape)


## 6. Compare naive full KV cache and MLA stored representation

실제 DeepSeek dimensions와 kernel absorption은 더 복잡하지만, 저장 관점의 차이는 작은 tensor에서도 볼 수 있다. full MHA는 token마다 모든 head의 K/V를 저장하는 반면 MLA는 compressed content latent와 작은 positional key만 저장하는 방향이다.


In [ ]:
full_kv_elements = (
    2
    * batch_size
    * sequence_length
    * num_heads
    * content_head_dim
)
mla_cached_elements = (
    compressed_kv.numel()
    + batch_size * sequence_length * rope_head_dim
)

print("naive full KV elements:", full_kv_elements)
print("MLA latent + positional cache elements:", mla_cached_elements)


## References and provenance

**MHA** — Vaswani et al. independent per-head Q/K/V baseline을 반영했다.

**MQA** — Shazeer, *Fast Transformer Decoding*. one shared KV head를 반영했다.

**GQA** — Ainslie et al. grouped KV heads를 반영했다.

**MLA** — DeepSeek-V2/V3. shared low-rank KV latent, head-specific content reconstruction, decoupled RoPE key/query subspace, compressed cache의 핵심을 반영했다. 실제 DeepSeek MLA는 query low-rank compression과 weight absorption 등 추가 최적화도 포함한다.
